# NVIDIA ALCHEMI: Large-Scale OLED Material Screening

Organic light-emitting diodes (OLEDs) power the displays of modern smartphones, TVs, and wearables.  Discovering **thermally stable** organic semiconductors — hosts, hole-transport layers (HTL), electron-transport layers (ETL), and hole-blocking layers (HBL) — is critical for device longevity.

This playbook demonstrates an **end-to-end computational screening workflow** inspired by [Universal Display Corporation's (UDC) collaboration with NVIDIA](https://blogs.nvidia.com/blog/udc-oled-generative-ai-drug-discovery/):

1. **Define** a candidate pool of 5 real OLED molecules from SMILES strings
2. **Generate** conformers via RDKit ETKDG, then **optimise** them concurrently with the **BGR NIM**
3. **Filter & deduplicate** conformers by energy and Kabsch-aligned RMSD
4. **Screen** the best conformer per molecule with **NVT molecular dynamics** at 500 K via the **BMD NIM**
5. **Rank** candidates by a composite thermal-stability score

All NIM calls are dispatched concurrently — NIMs support a fully asynchronous interface for dynamic batching and load balancing across GPUs

> **Companion playbook**: `alchemi-playbook-nims.ipynb` covers crystalline materials (NaCl MD, RDF, MSD, density, thermal expansion).

## What You Will Learn

- Define an OLED candidate pool from SMILES and validate with RDKit
- Generate conformers from SMILES using the ETKDGv3 algorithm
- Optimise conformer geometries concurrently via the BGR NIM
- Filter by energy and deduplicate rotamers with CREST-inspired RMSD comparison
- Run batched NVT MD at elevated temperature via the BMD NIM
- Rank molecules by composite stability metrics (energy, RMSD, bond integrity)

## Prerequisites

| Requirement | Details |
|---|---|
| BMD NIM | Running on `localhost:BMD_PORT` (default 8000) |
| BGR NIM | Running on `localhost:BGR_PORT` (default 8890) |
| **—OR—** | Cached responses in `cached_responses/` (provided) |
| Python packages | `ase`, `rdkit`, `pydantic`, `requests`, `matplotlib`, `pandas`, `numpy`, `aiohttp` |

Set `FAST_DEMO = True` (default) to run entirely from cached responses with no live endpoints required.

---
## 1. Environment Setup & Control Panel

In [ ]:
import importlib.metadata
import sys

from rdkit import __version__ as rdkit_version

print(f"Python {sys.version}")
for pkg in ["ase", "pydantic", "requests", "numpy", "matplotlib", "pandas", "aiohttp"]:
    print(f"  {pkg}: {importlib.metadata.version(pkg)}")
print(f"  rdkit: {rdkit_version}")

In [ ]:
# ── Control Panel ─────────────────────────────────────────────────────────
FAST_DEMO = False  # True = use cached responses; False = hit live endpoints

BMD_PORT = 8000
BGR_PORT = 8890
BMD_SERVER = f"http://localhost:{BMD_PORT}"
BGR_SERVER = f"http://localhost:{BGR_PORT}"

# MD parameters
T_SCREEN = 500.0  # Screening temperature (K) — mimics vacuum deposition
MD_TIME_PS = 10.0  # MD simulation time (ps)
DT_FS = 1.0  # Timestep (fs)
FRICTION = 1.0  # Langevin friction (ps^-1)
SAVE_INTERVAL = 100  # Save every N steps

# Conformer parameters
MAX_CONFORMERS = (
    5 if FAST_DEMO else 1000
)  # Max conformers per molecule (capped for demo)
BOX_SIZE = 50.0  # Vacuum box side length (Angstrom)
SEED = 42

# Directories
CACHE_DIR = "cached_responses/conformer-stability"
OUTPUT_DIR = "outputs"

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors

from helpers import (
    check_endpoint,
    run_md_or_load_cache,
    async_run_bgr_or_load_cache,
    async_run_md_or_load_cache,
    ase_to_atomic_data,
    ase_to_md_atomic_data,
    atomic_data_to_ase,
    MDConfig,
    extract_thermo_timeseries,
    pick_production_window,
    trajectory_to_ase_list,
    compute_rmsd,
    check_bond_integrity,
    compute_n_conformers,
    generate_conformers,
    filter_by_energy,
    deduplicate_conformers,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Helpers loaded.")

---
## 2. Endpoint Connectivity

In [ ]:
bmd_live = check_endpoint(BMD_SERVER) if not FAST_DEMO else False
bgr_live = check_endpoint(BGR_SERVER) if not FAST_DEMO else False

print(f"BMD NIM ({BMD_SERVER}): {'LIVE' if bmd_live else 'offline (using cache)'}")
print(f"BGR NIM ({BGR_SERVER}): {'LIVE' if bgr_live else 'offline (using cache)'}")

In [ ]:
from helpers import MDAtomicData, MDRequest

h2_atoms = MDAtomicData(
    coord=[0.0, 0.0, 0.0, 0.0, 0.0, 0.74],
    numbers=[1, 1],
    cell=[50.0, 0, 0, 0, 50.0, 0, 0, 0, 50.0],
    pbc=[True, True, True],
)
h2_config = MDConfig(temperature=300.0, dt=0.5, md_time_max=1.0, save_interval=10)

h2_reply = run_md_or_load_cache(
    h2_atoms, h2_config, BMD_SERVER, CACHE_DIR, "h2_hello_world", bmd_live
)
print(f"H2 hello-world: {h2_reply.status}, {len(h2_reply.trajectory)} frames")

In [ ]:
from IPython.display import JSON

# --- Pretty-print request and first response snapshot ---
request_dict = MDRequest(atoms=h2_atoms, config=h2_config).model_dump()
print("=== BMD REQUEST ===")
JSON(request_dict)

In [ ]:
print("\n=== RESPONSE FIRST MD SNAPSHOT ===")
md_snapshot_dict = h2_reply.trajectory[0].model_dump()
print(f"Status: {h2_reply.status}")
print(f"Frames: {len(h2_reply.trajectory)}")
print(f"Final energy: {h2_reply.trajectory[-1].energy:.4f} eV")
JSON(md_snapshot_dict)

---
## 3. OLED Candidate Pool

We screen five widely-used OLED materials spanning different device layers:

| Material | Role | Description |
|----------|------|-------------|
| **CBP** | Host | 4,4'-Bis(N-carbazolyl)-1,1'-biphenyl |
| **NPB** | HTL | N,N'-Di(1-naphthyl)-N,N'-diphenylbenzidine |
| **mCP** | Host | 1,3-Bis(N-carbazolyl)benzene |
| **BCP** | HBL | Bathocuproine (2,9-dimethyl-4,7-diphenyl-1,10-phenanthroline) |
| **TPBi** | ETL | 1,3,5-Tris(1-phenyl-1H-benzimidazol-2-yl)benzene |

In [ ]:
OLED_CANDIDATES = {
    "CBP": {
        "smiles": "c1ccc(-c2ccc(-n3c4ccccc4c4ccccc43)cc2)cc1",
        "role": "Host",
        "desc": "4,4'-Bis(N-carbazolyl)-1,1'-biphenyl",
    },
    "NPB": {
        "smiles": "c1ccc(-c2ccc(-N(c3ccccc3)c3ccc4ccccc4c3)cc2)cc1",
        "role": "HTL",
        "desc": "N,N'-Di(1-naphthyl)-N,N'-diphenylbenzidine",
    },
    "mCP": {
        "smiles": "c1ccc2c(c1)c1ccccc1n2-c1cccc(-n2c3ccccc3c3ccccc32)c1",
        "role": "Host",
        "desc": "1,3-Bis(N-carbazolyl)benzene",
    },
    "BCP": {
        "smiles": "Cc1ccc2c(-c3ccccc3)c3ccc4cc(-c5ccccc5)c(C)nc4c3nc2c1",
        "role": "HBL",
        "desc": "Bathocuproine",
    },
    "TPBi": {
        "smiles": "c1ccc(-c2nc3ccccc3[nH]2)cc1",
        "role": "ETL",
        "desc": "1-Phenyl-1H-benzimidazole (TPBi monomer unit)",
    },
}

mols = {}
for name, info in OLED_CANDIDATES.items():
    mol = Chem.MolFromSmiles(info["smiles"])
    assert mol is not None, f"Failed to parse {name}"
    mols[name] = mol
    print(f"  {name:5s}: {info['smiles'][:50]:50s}  OK")

print(f"\nAll {len(mols)} SMILES validated successfully.")

In [ ]:
grid = Draw.MolsToGridImage(
    list(mols.values()),
    molsPerRow=3,
    subImgSize=(350, 300),
    legends=[f"{n} ({OLED_CANDIDATES[n]['role']})" for n in mols],
)
grid

In [ ]:
from rdkit.Chem import Descriptors

rows = []
for name, mol in mols.items():
    mol_h = Chem.AddHs(mol)
    rows.append(
        {
            "Name": name,
            "Role": OLED_CANDIDATES[name]["role"],
            "Formula": rdMolDescriptors.CalcMolFormula(mol_h),
            "Atoms (w/ H)": mol_h.GetNumAtoms(),
            "MW (g/mol)": round(Descriptors.ExactMolWt(mol_h), 1),
            "Rotatable bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        }
    )

summary_df = pd.DataFrame(rows)
summary_df

---
## 4. Conformer Search: RDKit ETKDG + BGR Optimisation

The workflow proceeds in four stages:

1. **Generate** initial conformers per molecule using `EmbedMultipleConfs` (ETKDGv3)
   - Heuristic count: `min(1000, max(200, 3^n_rotatable_bonds))` — capped at `MAX_CONFORMERS` for this demo
2. **Optimise** all conformers concurrently via the BGR NIM (one async request per molecule)
3. **Filter** by energy — discard conformers > 3 kcal/mol above the minimum
4. **Deduplicate** rotamers via CREST-inspired Kabsch-aligned RMSD comparison (threshold 0.125 A)

\![UDC workflow diagram](assets/udc_workflow_diagram.png)

*Figure: UDC-inspired screening pipeline — SMILES → conformer generation → geometry optimisation → MD thermal stability → ranking.*

In [ ]:
import ase

all_conformers = {}  # name -> list of ASE Atoms in vacuum boxes
conf_table = []

for name, mol in mols.items():
    n_target = min(MAX_CONFORMERS, compute_n_conformers(mol))
    mol_3d = generate_conformers(mol, n_confs=n_target, seed=SEED, rmsd_threshold=0.05)
    n_embedded = mol_3d.GetNumConformers()

    # Convert each conformer to ASE Atoms in a vacuum box
    conformers_ase = []
    for cid in range(n_embedded):
        conf = mol_3d.GetConformer(cid)
        pos = np.array(conf.GetPositions())
        numbers = [a.GetAtomicNum() for a in mol_3d.GetAtoms()]
        centroid = pos.mean(axis=0)
        centred = pos - centroid + BOX_SIZE / 2.0

        atoms = ase.Atoms(numbers=numbers, positions=centred)
        atoms.set_cell([BOX_SIZE, BOX_SIZE, BOX_SIZE])
        atoms.set_pbc(True)
        conformers_ase.append(atoms)

    all_conformers[name] = conformers_ase
    n_rot = rdMolDescriptors.CalcNumRotatableBonds(mol)
    conf_table.append(
        {
            "Molecule": name,
            "Rotatable bonds": n_rot,
            "Requested": n_target,
            "Embedded": n_embedded,
        }
    )

pd.DataFrame(conf_table)

In [ ]:
import aiohttp
import asyncio

bgr_results = {}  # name -> BGRReply
t0 = time.time()


async def _run_all_bgr():
    connector = aiohttp.TCPConnector(limit=5)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = []
        for name, conformers_ase in all_conformers.items():
            atoms_list = [
                ase_to_atomic_data(a, _id=f"{name}_c{i}")
                for i, a in enumerate(conformers_ase)
            ]
            label = f"oled_{name.lower()}_bgr_confs"
            tasks.append(
                (
                    name,
                    async_run_bgr_or_load_cache(
                        atoms_list,
                        BGR_SERVER,
                        session,
                        CACHE_DIR,
                        label,
                        bgr_live,
                        timeout=300,
                    ),
                )
            )
        names = [name for name, _ in tasks]
        results = await asyncio.gather(*[coro for _, coro in tasks])
        for name, result in zip(names, results):
            bgr_results[name] = result


await _run_all_bgr()
wall_time = time.time() - t0

total_confs = sum(len(r.atoms) for r in bgr_results.values())
print(f"\nBGR optimisation complete: {total_confs} conformers in {wall_time:.1f}s")
for name, reply in bgr_results.items():
    n = len(reply.atoms)
    print(f"  {name:5s}: {n} optimised conformers")

In [ ]:
from helpers.conformers import filter_by_energy
from helpers.models import atomic_data_to_ase

filtered = {}  # name -> (indices, energies, coords_list)

for name, reply in bgr_results.items():
    all_e = np.array([a.energy for a in reply.atoms])
    mask = filter_by_energy(all_e, threshold_kcal=3.0)
    idxs = np.where(mask)[0].tolist()
    energies = all_e[mask]
    coords_list = [np.array(reply.atoms[j].coord).reshape(-1, 3) for j in idxs]
    filtered[name] = (idxs, energies, coords_list)
    print(f"  {name:5s}: {len(idxs)}/{len(all_e)} conformers within 3 kcal/mol")

print(f"\nTotal filtered: {sum(len(v[0]) for v in filtered.values())}")

In [ ]:
unique = {}  # name -> list of BGR atom indices (into bgr_results[name].atoms)

dedup_table = []
for name, (idxs, energies, coords_list) in filtered.items():
    if len(idxs) <= 1:
        unique_local = list(range(len(idxs)))
    else:
        unique_local = deduplicate_conformers(
            coords_list, energies, rmsd_threshold=0.125
        )
    # Map back to original BGR indices
    unique[name] = [idxs[j] for j in unique_local]
    dedup_table.append(
        {
            "Molecule": name,
            "After energy filter": len(idxs),
            "Unique": len(unique_local),
            "Duplicates removed": len(idxs) - len(unique_local),
        }
    )

pd.DataFrame(dedup_table)

In [ ]:
# Verify bond connectivity of unique conformers against SMILES reference
integrity_table = []
for name, u_idxs in unique.items():
    reply = bgr_results[name]
    # Build ASE frames from unique conformers
    frames = [atomic_data_to_ase(reply.atoms[i]) for i in u_idxs]
    if len(frames) >= 2:
        result = check_bond_integrity(frames, ref_frame_idx=0)
    else:
        result = {"n_broken": 0, "n_formed": 0, "integrity_score": 1.0}
    integrity_table.append(
        {
            "Molecule": name,
            "Conformers": len(frames),
            "Broken bonds": result["n_broken"],
            "Formed bonds": result["n_formed"],
            "Integrity": f"{result['integrity_score']:.2f}",
        }
    )

pd.DataFrame(integrity_table)

In [ ]:
best_conformers = {}  # name -> ASE Atoms (best conformer)
best_energies = {}

selection_table = []
for name, u_idxs in unique.items():
    reply = bgr_results[name]
    energies = np.array([reply.atoms[i].energy for i in u_idxs])
    best_local = np.argmin(energies)
    best_idx = u_idxs[best_local]
    best_atoms = atomic_data_to_ase(reply.atoms[best_idx])
    # Ensure vacuum box
    best_atoms.set_cell([BOX_SIZE, BOX_SIZE, BOX_SIZE])
    best_atoms.set_pbc(True)
    # Centre in box
    centroid = best_atoms.positions.mean(axis=0)
    best_atoms.positions += BOX_SIZE / 2.0 - centroid

    best_conformers[name] = best_atoms
    best_energies[name] = energies[best_local]

    total_gen = len(all_conformers[name])
    n_filt = len(filtered[name][0])
    n_uniq = len(u_idxs)
    selection_table.append(
        {
            "Molecule": name,
            "Generated": total_gen,
            "Post-filter": n_filt,
            "Unique": n_uniq,
            "Best E (eV)": f"{energies[best_local]:.3f}",
        }
    )

pd.DataFrame(selection_table)

In [ ]:
from helpers.constants import KCAL_MOL_TO_EV

role_colors = {"Host": "#2196F3", "HTL": "#FF9800", "HBL": "#9C27B0", "ETL": "#4CAF50"}
names = list(bgr_results.keys())

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, name in enumerate(names):
    ax = axes[idx]
    reply = bgr_results[name]
    all_e = np.array([a.energy for a in reply.atoms])
    role = OLED_CANDIDATES[name]["role"]
    color = role_colors[role]

    # All conformers (grey)
    ax.scatter(range(len(all_e)), all_e, c="lightgrey", s=20, zorder=1, label="All")

    # Unique conformers (coloured)
    u_idxs = unique[name]
    u_e = np.array([reply.atoms[j].energy for j in u_idxs])
    ax.scatter(u_idxs, u_e, c=color, s=40, zorder=2, label=f"Unique ({len(u_idxs)})")

    # Best (green star)
    best_idx = u_idxs[np.argmin(u_e)]
    ax.scatter(
        best_idx,
        best_energies[name],
        c="green",
        marker="*",
        s=200,
        zorder=3,
        label="Best",
    )

    # Energy cutoff threshold
    e_min = all_e.min()
    ax.axhline(
        e_min + 3.0 * KCAL_MOL_TO_EV,
        color="red",
        linestyle="--",
        linewidth=0.8,
        label="3 kcal/mol cutoff",
    )

    ax.set_title(f"{name} ({role})")
    ax.set_ylabel("BGR Energy (eV)")
    ax.set_xlabel("Conformer index")
    ax.legend(fontsize=7, loc="upper right")

# Hide the unused 6th subplot
axes[5].set_visible(False)

fig.suptitle("Conformer Energy Landscape", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "conformer_energy_landscape.png"), dpi=150)
plt.show()

In [ ]:
md_inputs = {}
for name, atoms in best_conformers.items():
    md_inputs[name] = ase_to_md_atomic_data(atoms)
    n = len(atoms)
    print(f"  {name:5s}: {n} atoms in {BOX_SIZE:.0f} A vacuum box")

In [ ]:
md_results = {}  # name -> MDReply
t0 = time.time()


async def _run_all_md():
    connector = aiohttp.TCPConnector(limit=5)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = []
        for name, mda in md_inputs.items():
            cfg = MDConfig(
                temperature=T_SCREEN,
                dt=DT_FS,
                nvt=True,
                npt=False,
                friction=FRICTION,
                md_time_max=MD_TIME_PS,
                save_interval=SAVE_INTERVAL,
            )
            label = f"oled_{name.lower()}_nvt_{int(T_SCREEN)}K"
            tasks.append(
                (
                    name,
                    async_run_md_or_load_cache(
                        mda,
                        cfg,
                        BMD_SERVER,
                        session,
                        CACHE_DIR,
                        label,
                        bmd_live,
                        timeout=1800,
                    ),
                )
            )
        names = [name for name, _ in tasks]
        results = await asyncio.gather(*[coro for _, coro in tasks])
        for name, result in zip(names, results):
            md_results[name] = result


await _run_all_md()
wall_time = time.time() - t0

print(f"\nMD screening complete in {wall_time:.1f}s")
for name, reply in md_results.items():
    print(f"  {name:5s}: {reply.status}, {len(reply.trajectory)} frames")

---
## 6. Stability Analysis & Ranking

We evaluate each molecule on four metrics computed over the **production window** (last 70% of the trajectory):

| Metric | Description | Weight |
|--------|-------------|--------|
| `E_std` | Energy fluctuation (eV) — lower = more stable | 10.0 |
| `E_drift` | Energy drift rate (eV/ps) — lower = better equilibration | 1000.0 |
| `RMSD_mean` | Mean structural deviation from reference (A) — lower = more rigid | 2.0 |
| `Bond integrity` | Fraction of frames with correct bond connectivity — higher = better | 50.0 (penalty) |

**Composite score** = `E_std * 10 + |E_drift| * 1000 + RMSD_mean * 2 + (1 - bond_integrity) * 50`

Lower composite score = more thermally stable.

In [ ]:
ranking_rows = []

for name, reply in md_results.items():
    mda = md_inputs[name]
    thermo = extract_thermo_timeseries(mda, reply.trajectory)
    s0, s1 = pick_production_window(thermo)

    prod_e = thermo["e_pot_eV"][s0:s1]
    e_mean = float(prod_e.mean())
    e_std = float(prod_e.std())
    t_arr = thermo["time_ps"][s0:s1]
    drift = float(np.polyfit(t_arr, prod_e, 1)[0]) if len(prod_e) > 1 else 0.0

    # RMSD
    frames = trajectory_to_ase_list(mda, reply.trajectory)
    _, rmsds = compute_rmsd(frames, start_frame=s0)
    rmsd_mean = float(rmsds.mean())

    # Bond integrity
    bi = check_bond_integrity(frames[s0:s1], ref_frame_idx=0)
    bond_score = bi["integrity_score"]

    composite = (
        e_std * 10.0 + abs(drift) * 1000.0 + rmsd_mean * 2.0 + (1.0 - bond_score) * 50.0
    )

    ranking_rows.append(
        {
            "Molecule": name,
            "Role": OLED_CANDIDATES[name]["role"],
            "E_mean (eV)": round(e_mean, 3),
            "E_std (eV)": round(e_std, 4),
            "E_drift (eV/ps)": f"{drift:.2e}",
            "RMSD_mean (A)": round(rmsd_mean, 3),
            "Bond integrity": round(bond_score, 3),
            "Composite": round(composite, 4),
        }
    )

ranking_df = pd.DataFrame(ranking_rows).sort_values("Composite")
ranking_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, reply in md_results.items():
    mda = md_inputs[name]
    frames = trajectory_to_ase_list(mda, reply.trajectory)
    s0, _ = pick_production_window(extract_thermo_timeseries(mda, reply.trajectory))
    times, rmsds = compute_rmsd(frames, start_frame=s0)
    role = OLED_CANDIDATES[name]["role"]
    ax.plot(times, rmsds, label=f"{name} ({role})", color=role_colors[role], alpha=0.8)

ax.set_xlabel("Time (ps)")
ax.set_ylabel("RMSD (A)")
ax.set_title(f"Structural RMSD — NVT {T_SCREEN:.0f} K")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "oled_rmsd_evolution.png"), dpi=150)
plt.show()

In [ ]:
# Normalised metrics comparison
metrics = ["E_std (eV)", "RMSD_mean (A)", "Bond integrity"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, metrics):
    vals = ranking_df.set_index("Molecule")[metric].astype(float)
    colors = [role_colors[OLED_CANDIDATES[n]["role"]] for n in vals.index]
    vals.plot.bar(ax=ax, color=colors)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "oled_metrics_comparison.png"), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

names = ranking_df["Molecule"].tolist()
scores = ranking_df["Composite"].tolist()
colors = [role_colors[OLED_CANDIDATES[n]["role"]] for n in names]
colors[0] = "#00C853"  # highlight best in green

ax.barh(names, scores, color=colors)
ax.set_xlabel("Composite Stability Score (lower = more stable)")
ax.set_title("OLED Material Stability Ranking")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "oled_ranking.png"), dpi=150)
plt.show()

In [ ]:
best_mol = ranking_df.iloc[0]
print("=" * 60)
print(f"MOST STABLE CANDIDATE: {best_mol['Molecule']}")
print(f"  Role:           {best_mol['Role']}")
print(f"  E_std:          {best_mol['E_std (eV)']} eV")
print(f"  RMSD_mean:      {best_mol['RMSD_mean (A)']} A")
print(f"  Bond integrity: {best_mol['Bond integrity']}")
print(f"  Composite:      {best_mol['Composite']}")
print("=" * 60)
print()
print("Full ranking (best to worst):")
for _, row in ranking_df.iterrows():
    print(
        f"  {row['Molecule']:5s} ({row['Role']:4s}): composite = {row['Composite']:.4f}"
    )

### Discussion

The ranking reflects known material properties from the OLED literature:

- **CBP** is a proven high-Tg (glass transition temperature ~62 °C) host material with excellent thermal stability, widely used in phosphorescent OLEDs.
- **mCP** is another carbazole-based host with good stability, commonly used for blue phosphorescent devices.
- **NPB** is the industry-standard hole-transport material, known for its morphological stability.
- **BCP** is a well-known hole-blocking material, though its smaller molecular size can lead to crystallisation at elevated temperatures.
- **TPBi** is a widely used electron-transport/hole-blocking material with good thermal properties.

The composite stability score captures a combination of energetic stability (low fluctuation, minimal drift), structural rigidity (low RMSD), and chemical integrity (no bond breaking).  In a production workflow, these results would be cross-validated with experimental Tg measurements and DFT calculations.

---
## 7. Extensions & Scaling

This demo screens 5 molecules at a single temperature.  The same infrastructure scales to production workflows:

- **Multi-temperature sweep**: Run MD at 300-700 K to estimate glass transition temperature (Tg) via density-temperature curves — the companion playbook (`alchemi-playbook-nims.ipynb`) demonstrates this for crystalline NaCl
- **Larger candidate pools**: The async batching pattern naturally extends to 100+ molecules — GPU throughput scales with batch size while wall time remains near-constant
- **Generative models**: Replace the hand-curated SMILES pool with candidates from generative models like [MolMIM](https://docs.nvidia.com/nim/bionemo/molmim/latest/overview.html) for de novo OLED material design
- **ALCHEMI toolkit-ops**: The [ALCHEMI Python toolkit](https://docs.nvidia.com/nim/alchemi/latest/toolkit-ops.html) provides higher-level abstractions for multi-stage workflows
- **DFT comparison**: Validate NIM-predicted energies against DFT (e.g. ORCA, Gaussian) for a subset of candidates
- **Agentic AI workflows**: Combine the NIM API with an LLM agent to autonomously navigate the design-simulate-analyse loop

## Summary

This playbook demonstrated:

1. **Candidate definition** — 5 real OLED molecules validated from SMILES
2. **Conformer generation** — RDKit ETKDGv3 with energy filtering and CREST-inspired deduplication
3. **BGR optimisation** — concurrent geometry relaxation via the NVIDIA ALCHEMI BGR NIM
4. **Thermal stability screening** — NVT MD at 500 K via the NVIDIA ALCHEMI BMD NIM
5. **Composite ranking** — multi-metric stability score combining energy, RMSD, and bond integrity

**Next steps**: See the companion playbook `alchemi-playbook-nims.ipynb` for crystalline material simulations, or extend this workflow with the scaling approaches described above.

## References

1. NVIDIA Developer Blog — [UDC and NVIDIA Accelerate OLED Material Discovery](https://blogs.nvidia.com/blog/udc-oled-generative-ai-drug-discovery/)
2. NVIDIA ALCHEMI NIM Documentation — [BMD NIM](https://docs.nvidia.com/nim/alchemi/latest/bmd.html) | [BGR NIM](https://docs.nvidia.com/nim/alchemi/latest/bgr.html)
3. Grimme, S. et al. — *Exploration of Chemical Compound, Conformer, and Reaction Space with Meta-Dynamics Simulations* (CREST method)
4. Landrum, G. — [RDKit: Open-Source Cheminformatics](https://www.rdkit.org/)